# Getting Historical Chain Constituents with LSEG Tick History REST API and LSEG Data Library for Python

The [LSEG Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) `Chain` object returns a chain's constituents, which you can then use with `get_data` and `get_history`. But some bonds chains change its constituent over time because exchanges periodically update which bonds meet the  maturity and coupon criteria for delivery. For instance, newly issued Treasuries are added while older bonds that have fallen below the minimum remaining maturity are removed. 

Examples:
- **0#TYc1=DLV**: CBoT 10-Year US Treasury Note
- **0#FGBMc1=DLV**: Eurex Euro-Bobl (5-year German Govt Bond)
- **0#JGBc1=DLV**: Osaka Exchange 10-Year Japanese Government Bond

See [README.md](./README.md) for more about these chain rules.

The `Chain` object returns **current constituents** because it gets data from the Real-Time platform. To resolve constituents for a specific time, use the [LSEG Tick History REST API](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api), which supports current and past constituents, including inactive, historical-only instruments. 

Once you get the constituents of that specific time from the Tick History, you can then retrieve their historical data with Tick History, the Data Library, or another LSEG API such as [Data Platform APIs (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis).

**Note**

You can also use the Tick History REST API for the entire workflow. See the [Tick History REST API tutorials](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorials-introduction) for details. This example is for developers who use the Data Library as their primary API.

In [44]:
import os
import time
import requests

## Prerequisite  

The Tick History - REST API requires the DataScope Select (DSS) credential and permission. Please contact your LSEG Account Manager or CSM team to help you get the DSS access credential for the Tick History API.

In [45]:
BASE_URL = "https://selectapi.datascope.lseg.com/RestApi/v1"
TIMEOUT = 30  # seconds

DSS_USERNAME = os.environ.get("DSS_USERNAME")
DSS_PASSWORD = os.environ.get("DSS_PASSWORD")

if not DSS_USERNAME or not DSS_PASSWORD:
    raise ValueError("Please set DSS_USERNAME and DSS_PASSWORD environment variables.")

## Authentication with DSS Server

First, request an authentication token from the DSS server. The request must include valid DSS credentials. For details, see [Tick History - REST API Tutorial 1: Connecting to the server](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-1-connecting-to-the-server).

The DSS authentication token is valid for 24 hours. Include it in the HTTP `Authorization` header as `Token {DSS Token}` in all Tick History data extraction requests.

This example uses a [Requests Session object](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) for all HTTP requests to DSS. A session reuses the TCP connection when making multiple requests to the same host, which can significantly improve performance. It also lets you set parameters, such as HTTP headers and cookies, that persist across requests.

In [46]:

# Reuse this session and its headers for Tick History requests.
session = requests.Session()
session.headers["Content-Type"] = "application/json"
session.headers["Prefer"] = "respond-async"

# Get an access token using the DSS credentials.
try:
    auth_response = session.post(
        f"{BASE_URL}/Authentication/RequestToken",
        json={"Credentials": {"Username": DSS_USERNAME, "Password": DSS_PASSWORD}},
        timeout=TIMEOUT,
    )
    auth_response.raise_for_status()
    print(f"Authentication Tick History response: {auth_response.status_code}")
    token = auth_response.json()["value"]
    if token:
        print("Token obtained")
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS authentication request failed") from exc

# Include the token in later API requests.
if token:
    session.headers["Authorization"] = f"Token {token}"

Authentication Tick History response: 200
Token obtained


## Resolving Historical Chain Constituents

The Tick History REST API's `/Search/HistoricalChainResolution` endpoint resolves the constituents of a Chain RIC for a specified date range. It supports currently active, or inactive 'historical only' instruments.

Notice that the `_session.post()` call does not include an `Authorization` header. We added the header to the `session` object during authentication, so Requests includes it automatically with every request made through that session. 

I define the historical chain constituent extraction in a `resolve_chain_request` function so we can call the endpoint multiple times throughout the workflow. Pass the `session` object to the function as `_session` each time.

You can find more detail regarding the `/Search/HistoricalChainResolution` endpoint from [Tick History - REST API Tutorial 15: Historical Chain Constituents Search](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-15-historical-chain-constituents-search).

In [50]:

def resolve_chain_request(request_session, base_url, chain_rics_list, start, end, timeout):
    """Return a dict of `{chain_ric: constituents}` for `chain_rics_list` between `start` and `end`."""
    print(f"Resolving {chain_rics_list} from {start} to {end}")

    payload = {
        "Request": {
            "ChainRics": chain_rics_list,
            "Range": {"Start": start, "End": end},
        }
    }

    try:
        historical_chain_response = request_session.post(
            f"{base_url}/Search/HistoricalChainResolution",
            json=payload,
            timeout=timeout,
        )

        # HTTP 202 = still processing; poll the monitor URL in the Location header until it returns 200
        while historical_chain_response.status_code == 202:
            print("Request is still processing, waiting 5 seconds...")
            time.sleep(5)
            historical_chain_response = request_session.get(historical_chain_response.headers["Location"], timeout=timeout)

        # Stop here if the API reports an error.
        historical_chain_response.raise_for_status()
        print(f"Historical Chain Resolution response: {historical_chain_response.status_code}")
        # Only a completed 200 response contains the constituent data.
        if historical_chain_response.status_code == 200:
            results = historical_chain_response.json().get("value", [])
            if not results:
                print(f"Historical Chain Resolution: No chain resolution results returned for {chain_rics_list}")
                return {}
            # Build a dict that maps each chain RIC to its list of constituents.
            # Example: {"0#TYc1=DLV": [...], "0#FGBMc1=DLV": [...]}
            constituents_by_chain = {chain["Identifier"]: chain.get("Constituents", []) for chain in results}
            for chain_ric, constituents in constituents_by_chain.items():
                print(f"Historical Chain Resolution: {chain_ric} - {len(constituents)} constituents returned")
            return constituents_by_chain
        return {}
    except requests.exceptions.RequestException as exc:
        raise RuntimeError("Historical chain resolution request failed") from exc


Lets make a first `/Search/HistoricalChainResolution` endpoint call via `resolve_chain_request` method. I am requesting *0#TYc1=DLV* (CBoT 10-Year US Treasury Note) and *#FGBMc1=DLV* (5-year German Govt Bond) RIC on the May and June 2024 period.

In [69]:
CHAIN_RICS = ["0#TYc1=DLV", "0#FGBMc1=DLV"]
START_2024 = "2024-05-01T00:00:00.000Z"
END_2024 = "2024-06-30T00:00:00.000Z"

# Fetch 2024 constituents via resolve_chain_request()
constituents_2024 = resolve_chain_request(
    request_session=session,
    base_url=BASE_URL,
    chain_rics_list=CHAIN_RICS,
    start=START_2024,
    end=END_2024,
    timeout=TIMEOUT
)

Resolving ['0#TYc1=DLV', '0#FGBMc1=DLV'] from 2024-05-01T00:00:00.000Z to 2024-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Historical Chain Resolution: 0#TYc1=DLV - 18 constituents returned
Historical Chain Resolution: 0#FGBMc1=DLV - 6 constituents returned


The response is a Python dictionary: each chain RIC is a key, and its value is the list of constituent records for that chain (`{"chain 1": [...], "chain 1": [...]}`). For example, `constituents_2024["0#TYc1=DLV"]` returns the constituent records for the CBoT 10-Year US Treasury chain.

The `get_valid_rics` function selects the requested chain's records, keeps only those with `"Status": "Valid"` and `"IdentifierType": "Ric"`, and returns their identifiers. Here, `us_10years_treasury` contains the valid RICs for `0#TYc1=DLV`.

In [70]:
def get_valid_rics(constituents,chain_ric):
    """Return identifiers marked valid and typed as RIC for the selected chain."""
    return [
        item["Identifier"]
        for item in constituents[chain_ric]
        if item["Status"] == "Valid" and item["IdentifierType"] == "Ric"
    ]

us_10years_treasury_2024 = get_valid_rics(constituents_2024, CHAIN_RICS[0])
us_10years_treasury_2024

['91282CBL4=',
 '91282CCB5=',
 '91282CCS8=',
 '91282CDJ7=',
 '91282CDY4=',
 '91282CEP2=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CJQ5=',
 '91282CJX0=',
 '91282CKC4=',
 '91282CKF7=',
 '91282CKN0=',
 '91282CKQ3=',
 '91282CKU4=',
 '91282CKW0=']

In [71]:
german_5years_bond_2024 = get_valid_rics(constituents_2024, CHAIN_RICS[1])
german_5years_bond_2024

['DE110246=', 'DE110247=', 'DE110249=', 'DE110262=', 'DEBU2502=', 'DEBU3502=']

A quick reminder: You can use the LSEG Tick History REST API for the entire workflow, including retrieving historical data for these constituents (`us_10years_treasury`,`german_5years_govtbond`), without using any other LSEG APIs. For example:
- [REST API Tutorial 6: On Demand intraday bars extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-6-on-demand-intraday-bars-extraction)
- [REST API Tutorial 4: On Demand tick data extraction](https://developers.lseg.com/en/api-catalog/refinitiv-tick-history/refinitiv-tick-history-rth-rest-api/tutorials#rest-api-tutorial-4-on-demand-tick-data-extraction)

The workflow below uses the LSEG Data Library with a Desktop Session to retrieve historical data for the same RICs and date range from Workspace. You can also use a Platform Session to retrieve data from the Data Platform ([see Data Library for Python QuickStart page](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python/quick-start)).

In [7]:
import lseg.data as ld 

In [8]:
ld.open_session()

<lseg.data.session.Definition object at 0x1dc84068830 {name='workspace'}>

I am demonstrating with Data Library `get_history` method to request historical data in a simplest way. You can see the example and full potential of this method from [EX-1.01.02-GetHistory.ipynb](https://github.com/LSEG-API-Samples/Example.DataLibrary.Python/blob/lseg-data-examples/Examples/1-Access/EX-1.01.02-GetHistory.ipynb) example code.

In [72]:
us_10years_treasury_2024_history = ld.get_history(universe=us_10years_treasury_2024[0:6], fields=["BID","ASK"], interval="weekly" , start=START_2024, end= END_2024)

us_10years_treasury_2024_history.head(20)

91282CBL4=            91282CCB5=            91282CCS8=             \
                  BID        ASK        BID        ASK        BID        ASK   
Date                                                                           
2024-05-03   80.78125     80.875  82.984375    83.0625   80.15625  80.210938   
2024-05-10  80.742188    80.8125  82.929688       83.0   80.09375  80.164062   
2024-05-17  81.164062      81.25  83.367188    83.4375    80.5625     80.625   
2024-05-24   80.84375  80.914062  83.054688  83.117188      80.25  80.304688   
2024-05-31  80.773438   80.84375  82.960938  83.023438  80.148438   80.21875   
2024-06-07   81.15625   81.21875  83.335938  83.398438  80.554688  80.609375   
2024-06-14  82.382812  82.476562   84.59375  84.695312  81.804688  81.929688   
2024-06-21   82.21875   82.28125   84.40625   84.46875   81.59375   81.65625   
2024-06-28  81.648438  81.703125  83.804688  83.867188   81.03125  81.078125   
2024-07-05  82.367188  82.460938  84.554688  84.648438  81.789062     81.875   

           91282CDJ7=            91282CDY4=            91282CEP2=             
                  BID        ASK        BID        ASK        BID        ASK  
Date                                                                          
2024-05-03  80.273438  80.320312    82.9375       83.0     89.125  89.179688  
2024-05-10      80.25    80.3125   82.90625   82.96875    89.0625  89.132812  
2024-05-17  80.742188  80.789062   83.40625   83.46875  89.601562  89.648438  
2024-05-24    80.4375  80.492188  83.101562   83.15625  89.273438  89.320312  
2024-05-31  80.320312     80.375  82.960938  83.015625   89.09375  89.148438  
2024-06-07   80.71875   80.78125   83.40625  83.445312  89.539062   89.59375  
2024-06-14       82.0     82.125  84.726562  84.820312  90.945312  91.039062  
2024-06-21    81.8125     81.875  84.539062   84.59375      90.75    90.8125  
2024-06-28  81.203125      81.25    83.8125     83.875   89.96875   90.03125  
2024-07-05  81.992188  82.046875  84.648438  84.710938    90.8125     90.875

Next, requesting daily historical data of *#FGBMc1=DLV* RIC with daily interval.

In [82]:
german_5years_govtbond_2024_history = ld.get_history(universe=german_5years_bond_2024, fields=["BID","ASK"], interval="daily" , start=START_2024, end= END_2024)

german_5years_govtbond_2024_history.head(20)

DE110246=         DE110247=         DE110249=         DE110262=  \
                 BID     ASK       BID     ASK       BID     ASK       BID   
Date                                                                         
2024-05-02    89.799  89.813    87.681  87.709    86.681  86.718    97.666   
2024-05-03    90.032  90.047    87.944  87.973    86.947  86.988    97.946   
2024-05-06    90.155   90.17    88.087  88.115    87.101  87.139    98.102   
2024-05-07    90.318  90.332    88.277  88.304    87.318  87.356    98.312   
2024-05-08    90.169  90.182    88.103   88.13    87.129  87.163    98.117   
2024-05-09    90.077  90.092    87.992  88.023    87.003  87.037    97.969   
2024-05-10    89.954  89.966    87.872  87.898    86.872  86.912    97.834   
2024-05-13    90.031  90.044    87.954  87.983    86.959  86.996    97.924   
2024-05-14    89.895  89.906    87.801  87.826     86.79  86.827    97.747   
2024-05-15    90.384  90.399    88.339  88.366    87.374  87.411    98.339   
2024-05-16    90.266  90.278    88.208  88.237    87.246  87.283    98.181   
2024-05-17    89.971  89.981    87.886  87.916    86.896  86.933    97.816   
2024-05-20    89.912  89.927    87.822  87.861    86.831   86.87    97.734   
2024-05-21    90.023  90.038    87.943  87.971    86.956  86.992    97.866   
2024-05-22    89.915  89.927    87.817  87.845     86.82  86.853     97.73   
2024-05-23    89.583  89.594    87.471  87.496    86.454  86.483    97.337   
2024-05-24    89.656  89.669    87.543  87.572    86.529  86.569    97.401   
2024-05-27    89.822  89.838    87.723  87.753    86.723  86.761    97.595   
2024-05-28    89.708  89.721    87.598   87.62    86.577  86.612    97.439   
2024-05-29    89.415  89.427    87.264   87.29    86.199  86.231    97.061   

                   DEBU2502=         DEBU3502=          
               ASK       BID     ASK       BID     ASK  
Date                                                    
2024-05-02  97.703    97.764  97.785    97.779  97.866  
2024-05-03  97.977    98.024  98.044    98.032  98.119  
2024-05-06  98.135    98.158  98.177    98.195   98.22  
2024-05-07  98.345    98.338  98.359    98.376  98.407  
2024-05-08  98.145    98.163   98.18    98.202  98.227  
2024-05-09  98.007    98.042   98.06    98.078   98.11  
2024-05-10  97.864     97.91  97.928    97.946  97.974  
2024-05-13  97.955    97.989  98.007    98.026  98.057  
2024-05-14  97.778    97.829  97.847    97.866    97.9  
2024-05-15   98.37    98.365  98.383    98.391  98.441  
2024-05-16  98.208    98.218  98.236    98.245  98.296  
2024-05-17  97.843    97.883    97.9    97.914  97.959  
2024-05-20  97.771     97.82  97.841    97.852  97.899  
2024-05-21  97.895    97.932  97.951    97.964  98.014  
2024-05-22  97.758    97.801   97.82     97.83  97.877  
2024-05-23  97.362     97.44  97.456     97.47  97.504  
2024-05-24   97.43    97.503  97.522    97.532  97.583  
2024-05-27  97.623    97.677  97.696    97.704  97.751  
2024-05-28  97.468    97.547  97.565    97.569  97.592  
2024-05-29   97.09    97.219  97.238    97.221  97.274

Lets try to request the same *0#TYc1=DLV* and *#FGBMc1=DLV* constituent RICs on the May and June 2026 period.

In [73]:
START_2026 = "2026-05-01T00:00:00.000Z"
END_2026   = "2026-06-30T00:00:00.000Z"

# Fetch 2026 constituents via resolve_chain_request()
constituents_2026 = resolve_chain_request(
    request_session = session, 
    base_url = BASE_URL, 
    chain_rics_list = CHAIN_RICS,
    start = START_2026, 
    end = END_2026, 
    timeout = TIMEOUT
)

Resolving ['0#TYc1=DLV', '0#FGBMc1=DLV'] from 2026-05-01T00:00:00.000Z to 2026-06-30T00:00:00.000Z
Historical Chain Resolution response: 200
Historical Chain Resolution: 0#TYc1=DLV - 19 constituents returned
Historical Chain Resolution: 0#FGBMc1=DLV - 5 constituents returned


In [75]:
us_10years_treasury_2026 = get_valid_rics(constituents_2026, CHAIN_RICS[0])
us_10years_treasury_2026

['91282CDJ7=',
 '91282CDY4=',
 '91282CFF3=',
 '91282CFV8=',
 '91282CGM7=',
 '91282CHC8=',
 '91282CHT1=',
 '91282CJJ1=',
 '91282CJZ5=',
 '91282CKQ3=',
 '91282CLF6=',
 '91282CPQ8=',
 '91282CPY1=',
 '91282CQC8=',
 '91282CQF1=',
 '91282CQN4=',
 '91282CQQ7=',
 '91282CQT1=',
 '91282CQW4=']

In [76]:
german_5years_bond_2026 = get_valid_rics(constituents_2026, CHAIN_RICS[1])
german_5years_bond_2026

['DE103073=', 'DE110253=', 'DE110256=', 'DE110258=', 'DEBU2506=']

### Comparing 2024 and 2026 Chain Constituents

Now let's compare the 2024 and 2026 constituent lists in a `pandas` DataFrame, with one row per bond RIC showing whether it was present in `2024`, `2026`, and its `Status` (`Removed`, `Unchanged`, or `Added`). Because deliverable bond baskets update over time—older bonds drop out once they fall below the minimum remaining maturity requirement while newly issued bonds are added—the constituent lists for May–June 2024 and May–June 2026 are different.

In [79]:
import pandas as pd


def compare_chain_constituents(rics_2024, rics_2026):
    """Compare 2024 and 2026 chain constituents in a one-row-per-RIC DataFrame."""
    # Use sets to find which RICs were removed, kept, or newly added.
    set_2024, set_2026 = set(rics_2024), set(rics_2026)
    removed = sorted(set_2024 - set_2026)
    unchanged = sorted(set_2024 & set_2026)
    added = sorted(set_2026 - set_2024)

    print(
        f"2024: {len(set_2024)} | 2026: {len(set_2026)} | "
        f"Removed: {len(removed)}, Unchanged: {len(unchanged)}, Added: {len(added)}"
    )

    # Group RICs by status (Removed -> Unchanged -> Added) with checkmarks for each period.
    groups = [
        (removed, "✓", "—", "Removed"),
        (unchanged, "✓", "✓", "Unchanged"),
        (added, "—", "✓", "Added"),
    ]
    rows = [
        {"RIC": ric, "2024": in_2024, "2026": in_2026, "Status": status}
        for rics, in_2024, in_2026, status in groups
        for ric in rics
    ]

    # Use RIC as the row label instead of 0, 1, 2... row numbers.
    return pd.DataFrame(rows, columns=["RIC", "2024", "2026", "Status"]).set_index("RIC")


compare_chain_constituents(us_10years_treasury_2024, us_10years_treasury_2026)

2024: 18 | 2026: 19 | Removed: 11, Unchanged: 7, Added: 12


,2024,2026,Status
RIC,,,
91282CBL4=,✓,—,Removed
91282CCB5=,✓,—,Removed
91282CCS8=,✓,—,Removed
91282CEP2=,✓,—,Removed
91282CJQ5=,✓,—,Removed
91282CJX0=,✓,—,Removed
91282CKC4=,✓,—,Removed
91282CKF7=,✓,—,Removed
91282CKN0=,✓,—,Removed


In [80]:
compare_chain_constituents(german_5years_bond_2024, german_5years_bond_2026)

2024: 6 | 2026: 5 | Removed: 6, Unchanged: 0, Added: 5


,2024,2026,Status
RIC,,,
DE110246=,✓,—,Removed
DE110247=,✓,—,Removed
DE110249=,✓,—,Removed
DE110262=,✓,—,Removed
DEBU2502=,✓,—,Removed
DEBU3502=,✓,—,Removed
DE103073=,—,✓,Added
DE110253=,—,✓,Added
DE110256=,—,✓,Added


Now I am using the Data Library `get_history` method to request historical data for *0#TYc1=DLV* RIC 2026's constituents using the same query as 2024.

In [83]:
us_10years_treasury_2026_history = ld.get_history(
    universe=us_10years_treasury_2026[0:6], 
    fields=["BID","ASK"], 
    interval="weekly", 
    start=START_2026, 
    end= END_2026
)

us_10years_treasury_2026_history.head(20)

91282CDJ7=            91282CDY4=            91282CFF3=             \
                  BID        ASK        BID        ASK        BID        ASK   
Date                                                                           
2026-05-01  86.664062   86.71875    88.5625     88.625  92.304688   92.34375   
2026-05-08  86.757812    86.8125   88.65625   88.71875     92.375    92.4375   
2026-05-15    85.6875  85.726562  87.523438  87.570312     91.125  91.171875   
2026-05-22  85.796875  85.859375  87.648438   87.71875  91.273438  91.335938   
2026-05-29     86.375  86.429688      88.25  88.289062  91.914062  91.960938   
2026-06-05  85.867188   85.90625  87.695312  87.757812    91.3125     91.375   
2026-06-12  86.164062   86.21875   88.03125  88.070312  91.664062  91.710938   
2026-06-19   86.15625  86.210938  88.007812  88.054688   91.65625  91.710938   
2026-06-26  86.632812   86.71875  88.492188  88.570312  92.179688  92.242188   
2026-07-03      86.25    86.3125   88.09375  88.140625  91.679688  91.726562   

           91282CFV8=            91282CGM7=            91282CHC8=             
                  BID        ASK        BID        ASK        BID        ASK  
Date                                                                          
2026-05-01  99.765625    99.8125  95.992188   96.03125  95.015625    95.0625  
2026-05-08  99.820312     99.875  96.070312     96.125   95.09375  95.132812  
2026-05-15   98.46875       98.5    94.6875  94.734375  93.671875  93.710938  
2026-05-22     98.625    98.6875  94.867188   94.90625  93.835938  93.890625  
2026-05-29  99.289062   99.34375   95.53125  95.570312  94.554688  94.585938  
2026-06-05     98.625    98.6875     94.875  94.929688  93.898438    93.9375  
2026-06-12   98.96875  99.007812   95.21875  95.257812      94.25   94.28125  
2026-06-19  98.953125       99.0  95.242188   95.28125  94.257812    94.3125  
2026-06-26  99.476562  99.554688   95.78125  95.851562  94.835938  94.890625  
2026-07-03  98.914062   98.96875  95.210938  95.257812      94.25   94.28125

And for the German 5 years government bond in 2026 with daily interval, we have:

In [84]:
german_5years_govtbond_2026_history = ld.get_history(
    universe=german_5years_bond_2026, 
    fields=["BID","ASK"], 
    interval="daily", 
    start=START_2026, 
    end= END_2026
)

german_5years_govtbond_2026_history.head(20)

DE103073=         DE110253=         DE110256=         DE110258=  \
                 BID     ASK       BID     ASK       BID     ASK       BID   
Date                                                                         
2026-05-04    86.479  86.504    87.775  87.802    86.444  86.473      85.1   
2026-05-05    86.607  86.635    87.914  87.939    86.588  86.609    85.251   
2026-05-06    86.965  87.035     88.26  88.307    86.967  86.996    85.653   
2026-05-07    86.982  87.016    88.266  88.296     86.96  86.984    85.643   
2026-05-08    86.929  86.953    88.208  88.251    86.903  86.933    85.573   
2026-05-11    86.744  86.779    88.042  88.072    86.722  86.746    85.385   
2026-05-12    86.488  86.518    87.802  87.835    86.474  86.491    85.121   
2026-05-13    86.396  86.421    87.739  87.766    86.377  86.397     85.04   
2026-05-14    86.695  86.718    88.013   88.04    86.682  86.696    85.363   
2026-05-15    86.276    86.3     87.64  87.665    86.265  86.288    84.905   
2026-05-18    86.291  86.315    87.651  87.676    86.269  86.294    84.909   
2026-05-19     86.17  86.194    87.537   87.57     86.15  86.177    84.777   
2026-05-20    86.605   86.63    87.936  87.969    86.589  86.609    85.235   
2026-05-21    86.554  86.581    87.888  87.914    86.529  86.552    85.165   
2026-05-22    86.834   86.86    88.137  88.167    86.815  86.838    85.484   
2026-05-25    87.251  87.285    88.516  88.549    87.216  87.262    85.909   
2026-05-26    87.079  87.104    88.347   88.38    87.051   87.08    85.726   
2026-05-27    87.124  87.149    88.396  88.426    87.099  87.129    85.773   
2026-05-28    87.247  87.271    88.506   88.54    87.227  87.254    85.896   
2026-05-29    87.375    87.4    88.621  88.651    87.351  87.378     86.04   

                   DEBU2506=          
               ASK       BID     ASK  
Date                                  
2026-05-04  85.129    98.604  98.619  
2026-05-05   85.28    98.744  98.752  
2026-05-06  85.683     99.13  99.146  
2026-05-07  85.672     99.11  99.127  
2026-05-08  85.619    99.048  99.063  
2026-05-11  85.431    98.852  98.872  
2026-05-12  85.151    98.589  98.601  
2026-05-13   85.07    98.493   98.51  
2026-05-14  85.387    98.791  98.804  
2026-05-15  84.934     98.37  98.382  
2026-05-18  84.939    98.359  98.373  
2026-05-19  84.806     98.24  98.249  
2026-05-20  85.265    98.675  98.684  
2026-05-21  85.195    98.602  98.616  
2026-05-22  85.504    98.883  98.897  
2026-05-25  85.929    99.289  99.303  
2026-05-26  85.756    99.103  99.117  
2026-05-27  85.802    99.144  99.157  
2026-05-28  85.926    99.254  99.267  
2026-05-29  86.069    99.374  99.387

To finish, I'll close the sessions used by both APIs. I can close the Data Library session with `ld.close_session()`; in the next step, I'll invalidate the Tick History token.

In [85]:
ld.close_session()

An application can revoke a Tick History REST API token using the `/Authentication/InvalidateToken` endpoint. After a successful request, the token can no longer be used.

In [86]:
try:
    # Ask Tick History to revoke the current token.
    logout_response = session.post(
        f"{BASE_URL}/Authentication/InvalidateToken",
        json={"Token": {"Token": token}},
        timeout=TIMEOUT,
    )
    print(f"Logout Tick History response: {logout_response.status_code}, {logout_response.text}")
    # Raise an error if the token could not be revoked.
    logout_response.raise_for_status()
except requests.exceptions.RequestException as exc:
    raise RuntimeError("DSS InvalidateToken request failed") from exc


Logout Tick History response: 200, {"@odata.context":"https://selectapi.datascope.lseg.com/RestApi/v1/$metadata#Edm.String","value":"Token has been invalidated successfully."}
